# Synthetic Product Dataset Generation

This notebook generates a synthetic built-in oven product portfolio using predifined product characteristics, generation probabilities, and configuration rules.

## Objective

The goal is to generate valid and realistic synthetic products rather than using unconstrained random combinations.

Generation logic:

1. Read master data from excel
2. Validate source tables
3. Sample driver characteristics
4. Apply configuration rules
5. Reject incompatible configurations
6. Generate valid products
7. Export PRODUCT_MASTER

In [1]:
import pandas as pd
import numpy as np

In [2]:
master_file = "../data/raw/oven_configurator_master_v1.0.xlsx"
excel_file = pd.ExcelFile(master_file)
excel_file.sheet_names

['Definition',
 'Proje Tanımlama',
 'PROJE TANIM',
 'Örnek Veri Seti',
 'Data Types',
 'DATASET_INFO',
 'README',
 'SYNTHETIC_GENERATION',
 'Dataset Architecture v1.0',
 'CHARACTERISTIC_DICTIONARY',
 'CHARACTERISTIC_VALUES',
 'CONFIGURATION_RULES',
 'PRODUCT_MASTER',
 'OVEN_FUNCTION_SET',
 'CUSTOMER_REQUEST',
 'REQUEST_GROUND_TRUTH']

In [3]:
characteristic_dictionary = pd.read_excel(
    master_file,
    sheet_name="CHARACTERISTIC_DICTIONARY", #pythonda sayma 0'dan başlar, notların bulunduğu ilk satırı hariç tutmak için header=1 kullanıyoruz
    header=1
)

characteristic_dictionary.head()



,CHARACTERISTIC,DATA_TYPE,ALLOWED_VALUES / RANGE,MATCH_METHOD,UNIT,DESCRIPTION
0,WIDTH_CM,NUMERIC,"45, 60, 70, 90",EXACT,cm,Product installation width
1,PACKAGING_TYPE,CATEGORICAL,"SHRINK, BOX",EXACT,-,Design of product packaging
2,PANEL_TYPE,CATEGORICAL,"GLASS, METAL",EXACT,-,Control panel material
3,PANEL_COLOR,CATEGORICAL,"BLACK, WHITE, SILVER, STAINLESS STEEL, DARK SI...",EXACT,-,Control panel color
4,POWER_CABLE,CATEGORICAL,"W/O CABLE, W/O PLUG, EU TYPE PLUG, US TYPE PLU...",EXACT,-,Power cable availability of product based on p...


In [4]:
print("shape:", characteristic_dictionary.shape)    #DataFrame boyutunu gösterir (satır sayısı, sütun sayısı)

print("\nColumns:")
print(characteristic_dictionary.columns.tolist())   #DataFrame sütun isimlerini liste olarak gösterir

print("\nDataFrame Info:")
characteristic_dictionary.info()                    #DataFrame hakkında genel bilgi verir

print("nMissing values:")
print(characteristic_dictionary.isna().sum())       #Her sütundaki eksik değer sayısını gösterir

shape: (24, 6)

Columns:
['CHARACTERISTIC', 'DATA_TYPE', 'ALLOWED_VALUES / RANGE', 'MATCH_METHOD', 'UNIT', 'DESCRIPTION']

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CHARACTERISTIC          24 non-null     object
 1   DATA_TYPE               24 non-null     object
 2   ALLOWED_VALUES / RANGE  24 non-null     object
 3   MATCH_METHOD            24 non-null     object
 4   UNIT                    24 non-null     object
 5   DESCRIPTION             24 non-null     object
dtypes: object(6)
memory usage: 1.3+ KB
nMissing values:
CHARACTERISTIC            0
DATA_TYPE                 0
ALLOWED_VALUES / RANGE    0
MATCH_METHOD              0
UNIT                      0
DESCRIPTION               0
dtype: int64


## Load and Validate Characteristic Values

The CHARACTERISTIC_VALUES table defines the valid values associated with each product characteristic.

This table will be validated aggainst the CHARACTERISTIC_DICTIONARY to ensure consistency between characteristic definitions and their available values.

In [5]:
characteristic_values = pd.read_excel(
    master_file,
    sheet_name="CHARACTERISTIC_VALUES",
    header=0
)

characteristic_values.head()

,CHARACTERISTIC,VALUE,DEFINITION
0,WIDTH_CM,V001,45
1,WIDTH_CM,V002,60
2,WIDTH_CM,V003,70
3,WIDTH_CM,V004,90
4,PACKAGING_TYPE,V001,SHRINK


In [6]:
print("Shape:", characteristic_values.shape)    #DataFrame boyutunu gösterir (satır sayısı, sütun sayısı)

print("\nColumns:")
print(characteristic_values.columns.tolist())   #DataFrame sütun isimlerini liste olarak gösterir

print("\nDataFrame Info:")
characteristic_values.info()                    #DataFrame hakkında genel bilgi verir

print("\nMissing Values:")
print(characteristic_values.isna().sum())       #Her sütundaki eksik değer sayısını gösterir

print("\nDuplicate Rows:")
print(characteristic_values.duplicated().sum()) #Her sütundaki yinelenen satır sayısını gösterir

Shape: (90, 3)

Columns:
['CHARACTERISTIC', 'VALUE', 'DEFINITION']

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CHARACTERISTIC  90 non-null     object
 1   VALUE           90 non-null     object
 2   DEFINITION      90 non-null     object
dtypes: object(3)
memory usage: 2.2+ KB

Missing Values:
CHARACTERISTIC    0
VALUE             0
DEFINITION        0
dtype: int64

Duplicate Rows:
0


In [7]:
dictionary_characteristics = set(
    characteristic_dictionary["CHARACTERISTIC"]         #Set of unique characteristics from the characteristic dictionary
)

values_characteristics = set(
    characteristic_values["CHARACTERISTIC"]             #Set of unique characteristics from the characteristic values
)

undefined_characteristics = (
    values_characteristics - dictionary_characteristics #Set of characteristics that are in the values but not in the dictionary
)

print("Characteristics in Dictionary:", len(dictionary_characteristics))    #Set of unique characteristics from the characteristic dictionary
print("Characteristics in Values:", len(values_characteristics))            #Set of unique characteristics from the characteristic values

print("\nUndefined Characteristics:")
print(undefined_characteristics)            #Set of characteristics that are in the values but not in the dictionary

missing_value_definitions = (
    dictionary_characteristics - values_characteristics #Set of characteristics that are in the dictionary but not in the values
)

print("\nCharacteristics Without Value Definitions:")
print(missing_value_definitions)        #Set of characteristics that are in the dictionary but not in the values

Characteristics in Dictionary: 24
Characteristics in Values: 24

Undefined Characteristics:
set()

Characteristics Without Value Definitions:
set()


## Load and Validate Configuration Rules

The CONFIGURATION_RULES table defines the domain constraints used to generate valid synthetic product configurations.

The rules will be validated against the characteristic and value master tables before they are used by the synthetic product generator.

In [8]:
configuration_rules = pd.read_excel(
    master_file,
    sheet_name="CONFIGURATION_RULES"
)

configuration_rules.head(10)

,RuleId,Type,Operator,CHARACTERISTIC,VALUE,DEFINITION
0,1,Auto Select,Action,TURNSPIT,V001,YES
1,1,AND,Rule,OVEN_FUNCTIONS,V004,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
2,1,AND,Rule,OVEN_FUNCTIONS,V007,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
3,2,Auto Select,Action,TURNSPIT,V002,NO
4,2,AND,Rule,OVEN_FUNCTIONS,V001,01-Top+Bottom | Bottom | Top
5,2,AND,Rule,OVEN_FUNCTIONS,V002,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...
6,2,AND,Rule,OVEN_FUNCTIONS,V003,03-Bottom | Top | Top+Bottom | Top+Bottom+Fan ...
7,2,AND,Rule,OVEN_FUNCTIONS,V005,05-OvenLight | Defrost | Top+Bottom | Top+Bott...
8,2,AND,Rule,OVEN_FUNCTIONS,V006,06-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
9,2,AND,Rule,OVEN_FUNCTIONS,V008,08-OvenLight | Defrost | Fan+Ring | Top+Bottom...


### Validate Rule Characterisic-Value Pairs

Each characteristic-value pair used in CONFIGURATION_RULES must exist in CHARACTERISTIC_VALUES.

This validation prevents configuration rules from referencing undefined characteristics or invalid values.

In [9]:
# Create valid (CHARACTERISTIC, VALUE) pairs from the master value table
valid_pairs = set(
    zip(
        characteristic_values["CHARACTERISTIC"],
        characteristic_values["VALUE"]
    )
)

# Create the pairs referenced by the configuration rules
rule_pairs = set(
    zip(
        configuration_rules["CHARACTERISTIC"],
        configuration_rules["VALUE"]
    )
)

# Find rule pairs that do not exist in CHARACTERISTIC_VALUES
invalid_rule_pairs = rule_pairs - valid_pairs

print("Valid characteristic-value pairs:", len(valid_pairs))
print("Characteristic-value pairs used in rules:", len(rule_pairs))

print("\nInvalid rule pairs:")
print(invalid_rule_pairs)

Valid characteristic-value pairs: 90
Characteristic-value pairs used in rules: 72

Invalid rule pairs:
set()


### Validate Configuration Rule Structure

The configuration rules are grouped by RuleId.

Each rule group may contain:
- condition rows (`Operator = Rule`)
- action rows (`Operator = Action`)

The purpose of this validation is to inspect whether each RuleId has the expected structure before implementing the rule engine.

In [10]:
# General structure of the rule table
print("Shape:", configuration_rules.shape)

print("\nNumber of unique RuleIds:")
print(configuration_rules["RuleId"].nunique())          #Number of unique RuleIds in the configuration rules

print("\nRule types:")
print(configuration_rules["Type"].value_counts())       #Count of each unique rule type in the configuration rules

print("\nOperators:")
print(configuration_rules["Operator"].value_counts())    #Count of each unique operator in the configuration rules

Shape: (170, 6)

Number of unique RuleIds:
37

Rule types:
Type
AND            116
Enable          21
Auto Select     20
INC             11
Disable          2
Name: count, dtype: int64

Operators:
Operator
Rule               116
Action              43
Incompatibility     11
Name: count, dtype: int64


### Inspect Rule Groups

Each RuleId represents a configuration rule group.

This step summarizes the internal structure of each rule before implementing the rule engine.

In [11]:
rule_summary = (
    configuration_rules
    .groupby("RuleId")
    .agg(
        ROW_COUNT=("RuleId", "size"),                                   #Count of rows for each RuleId
        RULE_ROWS=("Operator", lambda x: (x == "Rule").sum()),          #Count of rows with operator "Rule" for each RuleId
        ACTION_ROWS=("Operator", lambda x: (x == "Action").sum()),      #Count of rows with operator "Action" for each RuleId
        INC_ROWS=("Operator", lambda x: (x == "Incompatibility").sum()) #Count of rows with operator "Incompatibility" for each RuleId
    )
)

rule_summary

,ROW_COUNT,RULE_ROWS,ACTION_ROWS,INC_ROWS
RuleId,,,,
1,3,2,1,0
2,11,10,1,0
3,12,11,1,0
4,3,1,2,0
5,2,1,1,0
6,3,2,1,0
7,5,4,1,0
8,3,2,1,0
9,4,3,1,0


### Validate Rule Group Consistency

A RuleId must follow one of two valid structures:

1. Condition-Action rule:
   - one or more Rule rows
   - one or more Action rows
   - no Incompatibility rows

2. Incompatibility rule:
   - one or more Incompatibility rows
   - no Rule or Action rows

This validation ensures that incompatible rule structures are not mixed within the same RuleId.

In [12]:
# Identify RuleIds that mix incompatibility rows with normal Rule/Action rows

mixed_rule_groups = rule_summary[
    (rule_summary["INC_ROWS"] > 0)
    &
    (
        (rule_summary["RULE_ROWS"] > 0)
        | #OR
        (rule_summary["ACTION_ROWS"] > 0)
    )
]

print("Mixed rule groups:")
print(mixed_rule_groups)

Mixed rule groups:
Empty DataFrame
Columns: [ROW_COUNT, RULE_ROWS, ACTION_ROWS, INC_ROWS]
Index: []


### Validate Condition-Action Completeness

Every non-incompatibility rule group must contain:

- at least one condition row (`Operator = Rule`)
- at least one action row (`Operator = Action`)

This validation detects incomplete configuration rules before the rule engine is implemented.

In [13]:
# Select normal rule groups (non-incompatibility rules)
normal_rule_groups = rule_summary[
    rule_summary["INC_ROWS"] == 0
]

# Find normal rules with missing conditions or missing actions
incomplete_rule_groups = normal_rule_groups[
    (normal_rule_groups["RULE_ROWS"] == 0)
    |  #OR                                             
    (normal_rule_groups["ACTION_ROWS"] == 0)
]

print("Incomplete normal rule groups:")
print(incomplete_rule_groups)

Incomplete normal rule groups:
Empty DataFrame
Columns: [ROW_COUNT, RULE_ROWS, ACTION_ROWS, INC_ROWS]
Index: []


## Identify Driver and Rule-Controlled Characteristics

Before generating synthetic products, characteristics must be separated according to their role in the configuration system.

- **Driver characteristics:** selected during product generation and may trigger configuration rules.
- **Auto-selected characteristics:** deterministically assigned by Auto Select rules.
- **Enabled/Disabled characteristics:** their available values are constrained by rules.
- **Incompatibility characteristics:** participate in combinations that must be validated.

This separation prevents dependent characteristics from being sampled independently.

In [14]:
# Characteristics used as rule conditions
condition_characteristics = set(                                #set
    configuration_rules.loc[                                    #loc function is used to access a group of rows and columns by labels or a boolean array
        configuration_rules["Operator"] == "Rule",
        "CHARACTERISTIC"
    ]
)

# Characteristics controlled by actions
action_characteristics = set(                                   #set of unique characteristics from the
    configuration_rules.loc[                                    #loc function is used to access a group of rows and columns by labels or a boolean array
        configuration_rules["Operator"] == "Action",
        "CHARACTERISTIC"
    ]
)

# Characteristics participating in incompatibility rules
incompatibility_characteristics = set(                          #set of unique characteristics from the configuration rules with operator "Incompatibility"
    configuration_rules.loc[                                    #loc function is used to access a group of rows and columns by labels or a boolean array
        configuration_rules["Operator"] == "Incompatibility",
        "CHARACTERISTIC"
    ]
)

print("Condition characteristics:")
print(sorted(condition_characteristics))        #sorted by alphabetical order

print("\nAction-controlled characteristics:")
print(sorted(action_characteristics))

print("\nIncompatibility characteristics:")
print(sorted(incompatibility_characteristics))

Condition characteristics:
['HANDLE_TYPE', 'KNOB_TYPE', 'OVEN_CONTROL', 'OVEN_FUNCTIONS', 'PANEL_COLOR', 'WIDTH_CM']

Action-controlled characteristics:
['CLEANING', 'ENERGY_CLASS', 'HANDLE_COLOR', 'KNOB_COLOR', 'KNOB_STRUCTURE', 'NUMBER_OF_FUNCTIONS', 'OVEN_CONTROL', 'OVEN_TYPE', 'OVEN_VOLUME_L', 'PANEL_TYPE', 'PIZZA_TRAY', 'TURNSPIT', 'WIFI']

Incompatibility characteristics:
['BAKING_TRAY', 'HANDLE_COLOR', 'KNOB_COLOR', 'ROASTING_TRAY', 'WIRE_SHELF']


## Build Characteristic Role Matrix

A characteristic may participate in configuration logic in more than one role.

For each characteristic, we identify whether it is used as:

- **CONDITION**: used as an input condition in a rule
- **ACTION**: controlled or assigned by a rule action
- **INCOMPATIBILITY**: participates in an incompatible combination

A characteristic may have one, two, or all three roles.
This role matrix will help us understand the dependency structure before defining
driver and rule-controlled characteristics for synthetic product generation.

In [15]:
# Start with all characteristics defined in the dictionary
role_matrix = pd.DataFrame({
    "CHARACTERISTIC": sorted(dictionary_characteristics)
})

# Check whether each characteristic appears in each rule role
role_matrix["IS_CONDITION"] = role_matrix["CHARACTERISTIC"].isin(
    condition_characteristics
)

role_matrix["IS_ACTION"] = role_matrix["CHARACTERISTIC"].isin(
    action_characteristics
)

role_matrix["IS_INCOMPATIBILITY"] = role_matrix["CHARACTERISTIC"].isin(
    incompatibility_characteristics
)

role_matrix

,CHARACTERISTIC,IS_CONDITION,IS_ACTION,IS_INCOMPATIBILITY
0,BAKING_TRAY,False,False,True
1,CLEANING,False,True,False
2,ENERGY_CLASS,False,True,False
3,HANDLE_COLOR,False,True,True
4,HANDLE_TYPE,True,False,False
5,KNOB_COLOR,False,True,True
6,KNOB_STRUCTURE,False,True,False
7,KNOB_TYPE,True,False,False
8,NUMBER_OF_FUNCTIONS,False,True,False
9,OVEN_CONTROL,True,True,False


In [16]:
def get_roles(row):                         #define a function to get the roles of a characteristic based on the boolean columns in the role_matrix
    roles = []                              #function cerates an empty list called roles to store the roles of a characteristic based on the boolean columns in the role_matrix

    if row["IS_CONDITION"]:
        roles.append("CONDITION")           #append "CONDITION" to the roles list if the characteristic is a condition

    if row["IS_ACTION"]:
        roles.append("ACTION")              #append "ACTION" to the roles list if the characteristic is an action

    if row["IS_INCOMPATIBILITY"]:
        roles.append("INCOMPATIBILITY")     #append "INCOMPATIBILITY" to the roles list if the characteristic is an incompatibility

    if not roles:
        return "NONE"

    return " + ".join(roles)                #Join the roles in the list with " + " as a separator and return the resulting string


role_matrix["ROLE"] = role_matrix.apply(get_roles, axis=1)  #apply the get_roles function to each row of the role_matrix DataFrame and create a new column "ROLE" with the resulting role strings
                                                            #axis=1 means that the function is applied to each row (as opposed to each column)
role_matrix

,CHARACTERISTIC,IS_CONDITION,IS_ACTION,IS_INCOMPATIBILITY,ROLE
0,BAKING_TRAY,False,False,True,INCOMPATIBILITY
1,CLEANING,False,True,False,ACTION
2,ENERGY_CLASS,False,True,False,ACTION
3,HANDLE_COLOR,False,True,True,ACTION + INCOMPATIBILITY
4,HANDLE_TYPE,True,False,False,CONDITION
5,KNOB_COLOR,False,True,True,ACTION + INCOMPATIBILITY
6,KNOB_STRUCTURE,False,True,False,ACTION
7,KNOB_TYPE,True,False,False,CONDITION
8,NUMBER_OF_FUNCTIONS,False,True,False,ACTION
9,OVEN_CONTROL,True,True,False,CONDITION + ACTION


In [17]:
def classify_generation_role(row):

    if row["IS_ACTION"] and row["IS_CONDITION"]:
        return "INTERMEDIATE"

    elif row["IS_ACTION"]:
        return "DERIVED"

    elif row["IS_CONDITION"]:
        return "DRIVER"

    elif row["IS_INCOMPATIBILITY"]:
        return "VALIDATION_ONLY"

    else:
        return "INDEPENDENT"


role_matrix["INFERRED_GENERATION_ROLE"] = role_matrix.apply(     #INFERRED_GENERATION_ROLE column is created by applying the classify_generation_role function to each row of the role_matrix DataFrame
    classify_generation_role,
    axis=1
)

role_matrix[
    ["CHARACTERISTIC", "ROLE", "INFERRED_GENERATION_ROLE"]
]

,CHARACTERISTIC,ROLE,INFERRED_GENERATION_ROLE
0,BAKING_TRAY,INCOMPATIBILITY,VALIDATION_ONLY
1,CLEANING,ACTION,DERIVED
2,ENERGY_CLASS,ACTION,DERIVED
3,HANDLE_COLOR,ACTION + INCOMPATIBILITY,DERIVED
4,HANDLE_TYPE,CONDITION,DRIVER
5,KNOB_COLOR,ACTION + INCOMPATIBILITY,DERIVED
6,KNOB_STRUCTURE,ACTION,DERIVED
7,KNOB_TYPE,CONDITION,DRIVER
8,NUMBER_OF_FUNCTIONS,ACTION,DERIVED
9,OVEN_CONTROL,CONDITION + ACTION,INTERMEDIATE


## Build Characteristic Dependency Map

A dependency exists when a characteristic used as a rule condition controls
a characteristic used as an action within the same RuleId.

The dependency map is derived automatically from CONFIGURATION_RULES rather
than being manually defined.

This helps determine the correct generation order for synthetic products and
makes the architecture adaptable when configuration rules change.

In [18]:
dependency_pairs = []

for rule_id, group in configuration_rules.groupby("RuleId"):    #group the configuration_rules DataFrame by "RuleId" and iterate over each group, where rule_id is the unique RuleId and group is the corresponding subset of the DataFrame for that RuleId

    conditions = set(                                   #set of unique characteristics from the group where the operator is "Rule"
        group.loc[                                      #loc function is used to access a group of rows and columns by labels or a boolean array
            group["Operator"] == "Rule",
            "CHARACTERISTIC"
        ]
    )

    actions = set(                                      #set of unique characteristics from the group where the operator is "Action" 
        group.loc[                                      #loc function is used to access a group of rows and columns by labels or a boolean array  
            group["Operator"] == "Action",
            "CHARACTERISTIC"
        ]
    )

    for condition in conditions:
        for action in actions:
            dependency_pairs.append(
                {
                    "RuleId": rule_id,
                    "FROM_CHARACTERISTIC": condition,
                    "TO_CHARACTERISTIC": action
                }
            )

dependency_df = (
    pd.DataFrame(dependency_pairs)
    .drop_duplicates()
    .sort_values(
        ["FROM_CHARACTERISTIC", "TO_CHARACTERISTIC", "RuleId"]
    )
    .reset_index(drop=True)     #Reset the index of the DataFrame after sorting, and drop the old index to avoid adding it as a new column. drop=True means that the old index is not added as a new column in the DataFrame
)

dependency_df

,RuleId,FROM_CHARACTERISTIC,TO_CHARACTERISTIC
0,17,HANDLE_TYPE,HANDLE_COLOR
1,18,HANDLE_TYPE,HANDLE_COLOR
2,15,KNOB_TYPE,KNOB_COLOR
3,16,KNOB_TYPE,KNOB_COLOR
4,13,KNOB_TYPE,KNOB_STRUCTURE
5,14,KNOB_TYPE,KNOB_STRUCTURE
6,24,OVEN_CONTROL,ENERGY_CLASS
7,36,OVEN_CONTROL,PANEL_TYPE
8,37,OVEN_CONTROL,PANEL_TYPE
9,34,OVEN_CONTROL,WIFI


In [19]:
characteristic_dependencies = (
    dependency_df[          #define a new DataFrame characteristic_dependencies that contains only the unique pairs of FROM_CHARACTERISTIC and TO_CHARACTERISTIC from the dependency_df DataFrame
        ["FROM_CHARACTERISTIC", "TO_CHARACTERISTIC"]
    ]
    .drop_duplicates()
    .sort_values(
        ["FROM_CHARACTERISTIC", "TO_CHARACTERISTIC"]
    )
    .reset_index(drop=True)
)

print(
    "Unique characteristic dependencies:",
    len(characteristic_dependencies)
)

characteristic_dependencies

Unique characteristic dependencies: 16


,FROM_CHARACTERISTIC,TO_CHARACTERISTIC
0,HANDLE_TYPE,HANDLE_COLOR
1,KNOB_TYPE,KNOB_COLOR
2,KNOB_TYPE,KNOB_STRUCTURE
3,OVEN_CONTROL,ENERGY_CLASS
4,OVEN_CONTROL,PANEL_TYPE
5,OVEN_CONTROL,WIFI
6,OVEN_FUNCTIONS,CLEANING
7,OVEN_FUNCTIONS,ENERGY_CLASS
8,OVEN_FUNCTIONS,NUMBER_OF_FUNCTIONS
9,OVEN_FUNCTIONS,OVEN_CONTROL


## Determine Characteristic Generation Levels

The dependency map is used to determine the order in which characteristics
should be generated.

Characteristics with no parent dependencies can be generated first.
Dependent characteristics are generated only after their parent
characteristics are available.

In [20]:
# Characteristics that appear as dependency sources
parent_characteristics = set(
    characteristic_dependencies["FROM_CHARACTERISTIC"]
)

# Characteristics that appear as dependency targets
child_characteristics = set(
    characteristic_dependencies["TO_CHARACTERISTIC"]
)

print("Parent characteristics:")
print(sorted(parent_characteristics))

print("\nChild characteristics:")
print(sorted(child_characteristics))

Parent characteristics:
['HANDLE_TYPE', 'KNOB_TYPE', 'OVEN_CONTROL', 'OVEN_FUNCTIONS', 'PANEL_COLOR', 'WIDTH_CM']

Child characteristics:
['CLEANING', 'ENERGY_CLASS', 'HANDLE_COLOR', 'KNOB_COLOR', 'KNOB_STRUCTURE', 'NUMBER_OF_FUNCTIONS', 'OVEN_CONTROL', 'OVEN_TYPE', 'OVEN_VOLUME_L', 'PANEL_TYPE', 'PIZZA_TRAY', 'TURNSPIT', 'WIFI']


In [21]:
root_characteristics = (
    parent_characteristics - child_characteristics  #- means set difference, so this line calculates the set of characteristics that are parents but not children, which are considered root characteristics
)

print("Root characteristics:")
print(sorted(root_characteristics))

Root characteristics:
['HANDLE_TYPE', 'KNOB_TYPE', 'OVEN_FUNCTIONS', 'PANEL_COLOR', 'WIDTH_CM']


### Calculate Dependency Levels

Characteristics are assigned to dependency levels based on the rule graph.

- Level 0 contains root characteristics.
- A characteristic can be assigned to the next level only after all of its parent characteristics have already been assigned.
- This creates a valid generation order for dependent product characteristics.

In [22]:
# Start with root characteristics at Level 0
dependency_levels = {
    characteristic: 0
    for characteristic in root_characteristics
}

remaining_characteristics = (
    child_characteristics - set(dependency_levels.keys()) # keys() returns a view of the keys in the dependency_levels dictionary, which are the characteristics that have already been assigned a level. The set() function converts this view into a set, allowing for set operations. The difference operation (-) is then used to find characteristics that are in child_characteristics but not in the set of already assigned characteristics, resulting in a set of characteristics that still need to be assigned a level
)

current_level = 1

while remaining_characteristics:    #as long as there are characteristics that have not yet been assigned a level, the while loop continues to iterate. The loop will terminate when all characteristics have been assigned a level, i.e., when remaining_characteristics becomes empty

    assigned_this_round = set()     #set() creates an empty set called assigned_this_round to keep track of characteristics that are assigned a level in the current iteration of the while loop

    # Only characteristics assigned BEFORE this level are available
    previously_assigned = set(dependency_levels.keys())
    
    for characteristic in remaining_characteristics:

        # Find all direct parents of this characteristic
        parents = set(
            characteristic_dependencies.loc[    #loc means label-based indexing, so this line selects the rows in the characteristic_dependencies DataFrame where the TO_CHARACTERISTIC column matches the current characteristic and returns the values in the FROM_CHARACTERISTIC column
                characteristic_dependencies["TO_CHARACTERISTIC"] == characteristic,
                "FROM_CHARACTERISTIC"
            ]   # Select the rows where the TO_CHARACTERISTIC column matches the current characteristic and return the values in the FROM_CHARACTERISTIC column
        )

        # Assign the characteristic only if all its parents
        # have already been assigned to an earlier level
        if parents.issubset(previously_assigned):
            dependency_levels[characteristic] = current_level
            assigned_this_round.add(characteristic)

    # Safety check: prevents an infinite loop if the dependency
    # graph contains a cycle
    if not assigned_this_round:
        raise ValueError(
            "Dependency graph may contain a cycle or unresolved dependency."
        )

    remaining_characteristics -= assigned_this_round #-= means set difference, so this line removes the characteristics that have been assigned a level in this round from the remaining_characteristics set
    current_level += 1 #+= means increment, so this line increments the current_level variable by 1 for the next iteration of the while loop

dependency_levels

{'PANEL_COLOR': 0,
 'WIDTH_CM': 0,
 'OVEN_FUNCTIONS': 0,
 'KNOB_TYPE': 0,
 'HANDLE_TYPE': 0,
 'OVEN_VOLUME_L': 1,
 'TURNSPIT': 1,
 'OVEN_CONTROL': 1,
 'KNOB_STRUCTURE': 1,
 'OVEN_TYPE': 1,
 'NUMBER_OF_FUNCTIONS': 1,
 'PIZZA_TRAY': 1,
 'HANDLE_COLOR': 1,
 'CLEANING': 1,
 'KNOB_COLOR': 1,
 'ENERGY_CLASS': 2,
 'WIFI': 2,
 'PANEL_TYPE': 2}

### Inspect Multi-Level Dependencies

The dependency levels are inspected together with their parent characteristics
to understand why each dependent characteristic is assigned to a specific
generation level.

In [23]:
# Create a readable dependency-level summary

dependency_level_rows = []

for characteristic, level in dependency_levels.items():

    parents = sorted(
        characteristic_dependencies.loc[
            characteristic_dependencies["TO_CHARACTERISTIC"] == characteristic,
            "FROM_CHARACTERISTIC"
        ].unique()
    )

    dependency_level_rows.append(
        {
            "CHARACTERISTIC": characteristic,
            "LEVEL": level,
            "PARENTS": ", ".join(parents) if parents else "-"
        }
    )

dependency_level_df = pd.DataFrame(dependency_level_rows)

dependency_level_df = dependency_level_df.sort_values(
    ["LEVEL", "CHARACTERISTIC"]
).reset_index(drop=True)

dependency_level_df

,CHARACTERISTIC,LEVEL,PARENTS
0,HANDLE_TYPE,0,-
1,KNOB_TYPE,0,-
2,OVEN_FUNCTIONS,0,-
3,PANEL_COLOR,0,-
4,WIDTH_CM,0,-
5,CLEANING,1,OVEN_FUNCTIONS
6,HANDLE_COLOR,1,"HANDLE_TYPE, PANEL_COLOR"
7,KNOB_COLOR,1,"KNOB_TYPE, PANEL_COLOR"
8,KNOB_STRUCTURE,1,KNOB_TYPE
9,NUMBER_OF_FUNCTIONS,1,OVEN_FUNCTIONS


### Add Independent Characteristics to the Generation Plan

Characteristics that do not participate in the dependency graph can be
generated independently.

These characteristics are added to Level 0 because they do not require
another characteristic to be generated first.

In [25]:
# All characteristics defined in the master dictionary
all_characteristics = set(
    characteristic_dictionary["CHARACTERISTIC"]
)

# Characteristics participating anywhere in the dependency graph
graph_characteristics = (
    parent_characteristics | child_characteristics #| means set union, so this line combines the parent_characteristics and child_characteristics sets to create a new set containing all characteristics that are either parents or children in the dependency graph
)

# Characteristics outside the dependency graph
independent_characteristics = (
    all_characteristics - graph_characteristics # - means set difference, so this line calculates the set of characteristics that are in all_characteristics but not in graph_characteristics
)


print("Independent characteristics:")
print(sorted(independent_characteristics))

Independent characteristics:
['BAKING_TRAY', 'PACKAGING_TYPE', 'POWER_CABLE', 'ROASTING_TRAY', 'TELESCOPIC_RAILS', 'WIRE_SHELF']


## Final Characteristic Generation Plan

The dependency graph determines the generation order of dependent characteristics.

Characteristics outside the dependency graph are generated independently or
sampled before incompatibility validation.

The final generation plan combines:

- characteristic role,
- inferred generation role,
- dependency level,
- parent characteristics.

In [26]:
# Create the final generation plan from the inferred role matrix

generation_plan = role_matrix.copy()

generation_plan["LEVEL"] = generation_plan["CHARACTERISTIC"].map(
    dependency_levels
)

generation_plan["PARENTS"] = generation_plan["CHARACTERISTIC"].apply(
    lambda characteristic: ", ".join(
        sorted(
            characteristic_dependencies.loc[
                characteristic_dependencies["TO_CHARACTERISTIC"] == characteristic,
                "FROM_CHARACTERISTIC"
            ].unique()
        )
    )
)

generation_plan

,CHARACTERISTIC,IS_CONDITION,IS_ACTION,IS_INCOMPATIBILITY,ROLE,INFERRED_GENERATION_ROLE,LEVEL,PARENTS
0,BAKING_TRAY,False,False,True,INCOMPATIBILITY,VALIDATION_ONLY,NaN,
1,CLEANING,False,True,False,ACTION,DERIVED,1.0,OVEN_FUNCTIONS
2,ENERGY_CLASS,False,True,False,ACTION,DERIVED,2.0,"OVEN_CONTROL, OVEN_FUNCTIONS"
3,HANDLE_COLOR,False,True,True,ACTION + INCOMPATIBILITY,DERIVED,1.0,"HANDLE_TYPE, PANEL_COLOR"
4,HANDLE_TYPE,True,False,False,CONDITION,DRIVER,0.0,
5,KNOB_COLOR,False,True,True,ACTION + INCOMPATIBILITY,DERIVED,1.0,"KNOB_TYPE, PANEL_COLOR"
6,KNOB_STRUCTURE,False,True,False,ACTION,DERIVED,1.0,KNOB_TYPE
7,KNOB_TYPE,True,False,False,CONDITION,DRIVER,0.0,
8,NUMBER_OF_FUNCTIONS,False,True,False,ACTION,DERIVED,1.0,OVEN_FUNCTIONS
9,OVEN_CONTROL,True,True,False,CONDITION + ACTION,INTERMEDIATE,1.0,OVEN_FUNCTIONS


## Define Sampling Strategy

Synthetic products should not be generated using uniform random sampling.

Driver and independent characteristics are sampled using controlled probability
distributions to create a realistic but sufficiently diverse synthetic product portfolio.

The sampling strategy should:

- represent common product configurations more frequently,
- preserve less common alternatives,
- provide sufficient coverage for downstream analysis,
- avoid unrealistic uniform distributions.

In [40]:
# Define sampling weights for WIDTH_CM

sampling_weights = {
    "WIDTH_CM": {
        "V001": 0.10,
        "V002": 0.65,
        "V003": 0.10,
        "V004": 0.15
    }
}

# Validate that probabilities sum to 1

assert np.isclose(
    sum(sampling_weights["WIDTH_CM"].values()),
    1.0
), "WIDTH_CM sampling probabilities must sum to 1."

# Display sampling strategy together with master-data definitions

width_sampling = characteristic_values[
    characteristic_values["CHARACTERISTIC"] == "WIDTH_CM"
][["CHARACTERISTIC", "VALUE", "DEFINITION"]].copy()

width_sampling["PROBABILITY"] = width_sampling["VALUE"].map(
    sampling_weights["WIDTH_CM"]
)

width_sampling

,CHARACTERISTIC,VALUE,DEFINITION,PROBABILITY
0,WIDTH_CM,V001,45,0.10
1,WIDTH_CM,V002,60,0.65
2,WIDTH_CM,V003,70,0.10
3,WIDTH_CM,V004,90,0.15


In [41]:
oven_function_values = characteristic_values[
    characteristic_values["CHARACTERISTIC"] == "OVEN_FUNCTIONS"
][["VALUE", "DEFINITION"]].copy()

oven_function_values

,VALUE,DEFINITION
30,V001,01-Top+Bottom | Bottom | Top
31,V002,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...
32,V003,03-Bottom | Top | Top+Bottom | Top+Bottom+Fan ...
33,V004,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
34,V005,05-OvenLight | Defrost | Top+Bottom | Top+Bott...
35,V006,06-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
36,V007,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
37,V008,08-OvenLight | Defrost | Fan+Ring | Top+Bottom...
38,V009,09-Defrost | TURBO | Bottom | Top+Bottom | Top...
39,V010,10-OvenLight | Defrost | Fan+Ring | Top+Bottom...


In [ ]:
# Infer oven technology type from the function-set definition

def infer_function_type(definition):    #define a function to infer the oven technology type based on the definition of the function set. The function takes a string input (definition) and returns a string output representing the inferred function type. The function first converts the input definition to uppercase to ensure case-insensitive matching. It then checks for specific keywords in the definition to determine the appropriate function type. If "FAN+RING" is found, it returns "MULTIFUNCTION". If either "FAN" or "TURBO" is found, it returns "ELECTROTURBO". If none of these keywords are present, it defaults to returning "STATIC".
    definition_upper = definition.upper()

    if "FAN+RING" in definition_upper:
        return "MULTIFUNCTION"
    elif "FAN" in definition_upper or "TURBO" in definition_upper:
        return "ELECTROTURBO"
    else:
        return "STATIC"


oven_function_overview["FUNCTION_TYPE"] = (
    oven_function_overview["DEFINITION"]
    .apply(infer_function_type)
)

oven_function_overview[
    ["VALUE", "FUNCTION_SET", "FUNCTION_TYPE", "DEFINITION"]
]

,VALUE,FUNCTION_SET,FUNCTION_TYPE,DEFINITION
30,V001,01,STATIC,01-Top+Bottom | Bottom | Top
31,V002,02,ELECTROTURBO,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...
32,V003,03,ELECTROTURBO,03-Bottom | Top | Top+Bottom | Top+Bottom+Fan ...
33,V004,04,ELECTROTURBO,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
34,V005,05,ELECTROTURBO,05-OvenLight | Defrost | Top+Bottom | Top+Bott...
35,V006,06,MULTIFUNCTION,06-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
36,V007,07,MULTIFUNCTION,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
37,V008,08,MULTIFUNCTION,08-OvenLight | Defrost | Fan+Ring | Top+Bottom...
38,V009,09,ELECTROTURBO,09-Defrost | TURBO | Bottom | Top+Bottom | Top...
39,V010,10,MULTIFUNCTION,10-OvenLight | Defrost | Fan+Ring | Top+Bottom...


In [46]:
# Find RuleIds where OVEN_FUNCTIONS controls NUMBER_OF_FUNCTIONS

function_count_rule_ids = []

for rule_id, group in configuration_rules.groupby("RuleId"):

    has_oven_functions_rule = (
        (group["CHARACTERISTIC"] == "OVEN_FUNCTIONS")
        & (group["Operator"] == "Rule")
    ).any()

    has_function_count_action = (
        (group["CHARACTERISTIC"] == "NUMBER_OF_FUNCTIONS")
        & (group["Operator"] == "Action")
    ).any()

    if has_oven_functions_rule and has_function_count_action:
        function_count_rule_ids.append(rule_id)

function_count_rule_ids

[5, 6, 7, 8, 9]

In [47]:
# Build OVEN_FUNCTIONS -> NUMBER_OF_FUNCTIONS mapping from configuration rules

function_count_mapping = []

for rule_id in function_count_rule_ids:

    group = configuration_rules[
        configuration_rules["RuleId"] == rule_id
    ]

    # Get OVEN_FUNCTIONS values used as conditions
    oven_function_ids = group.loc[
        (group["CHARACTERISTIC"] == "OVEN_FUNCTIONS")
        & (group["Operator"] == "Rule"),
        "VALUE"
    ].tolist()

    # Get the NUMBER_OF_FUNCTIONS value selected by the action
    function_count_value_id = group.loc[
        (group["CHARACTERISTIC"] == "NUMBER_OF_FUNCTIONS")
        & (group["Operator"] == "Action"),
        "VALUE"
    ].iloc[0]

    # Convert the master-data VALUE ID into its readable definition
    function_count = characteristic_values.loc[
        (characteristic_values["CHARACTERISTIC"] == "NUMBER_OF_FUNCTIONS")
        & (characteristic_values["VALUE"] == function_count_value_id),
        "DEFINITION"
    ].iloc[0]

    # Create one mapping row for each OVEN_FUNCTIONS value
    for oven_function_id in oven_function_ids:
        function_count_mapping.append({
            "VALUE": oven_function_id,
            "FUNCTION_COUNT": function_count
        })


function_count_mapping = pd.DataFrame(function_count_mapping)

function_count_mapping

,VALUE,FUNCTION_COUNT
0,V001,3
1,V002,5
2,V003,5
3,V004,8
4,V005,8
5,V006,8
6,V007,8
7,V008,10
8,V009,10
9,V010,11


In [48]:
# Combine function-set definitions with derived function counts

oven_function_sampling = oven_function_overview.merge(
    function_count_mapping,
    on="VALUE",
    how="left"
)

# Create a compact human-readable label
oven_function_sampling["LABEL"] = (
    oven_function_sampling["FUNCTION_SET"]
    + "_"
    + oven_function_sampling["FUNCTION_COUNT"].astype(str)
    + "_"
    + oven_function_sampling["FUNCTION_TYPE"]
)

oven_function_sampling[
    [
        "VALUE",
        "FUNCTION_SET",
        "FUNCTION_COUNT",
        "FUNCTION_TYPE",
        "LABEL",
        "DEFINITION"
    ]
]

,VALUE,FUNCTION_SET,FUNCTION_COUNT,FUNCTION_TYPE,LABEL,DEFINITION
0,V001,01,3,STATIC,01_3_STATIC,01-Top+Bottom | Bottom | Top
1,V002,02,5,ELECTROTURBO,02_5_ELECTROTURBO,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...
2,V003,03,5,ELECTROTURBO,03_5_ELECTROTURBO,03-Bottom | Top | Top+Bottom | Top+Bottom+Fan ...
3,V004,04,8,ELECTROTURBO,04_8_ELECTROTURBO,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
4,V005,05,8,ELECTROTURBO,05_8_ELECTROTURBO,05-OvenLight | Defrost | Top+Bottom | Top+Bott...
5,V006,06,8,MULTIFUNCTION,06_8_MULTIFUNCTION,06-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
6,V007,07,8,MULTIFUNCTION,07_8_MULTIFUNCTION,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
7,V008,08,10,MULTIFUNCTION,08_10_MULTIFUNCTION,08-OvenLight | Defrost | Fan+Ring | Top+Bottom...
8,V009,09,10,ELECTROTURBO,09_10_ELECTROTURBO,09-Defrost | TURBO | Bottom | Top+Bottom | Top...
9,V010,10,11,MULTIFUNCTION,10_11_MULTIFUNCTION,10-OvenLight | Defrost | Fan+Ring | Top+Bottom...


In [50]:
# Define controlled sampling probabilities for OVEN_FUNCTIONS

oven_function_probabilities = {
    "V001": 0.04,
    "V002": 0.07,
    "V003": 0.07,
    "V004": 0.10,
    "V005": 0.12,
    "V006": 0.12,
    "V007": 0.10,
    "V008": 0.10,
    "V009": 0.08,
    "V010": 0.08,
    "V011": 0.06,
    "V012": 0.06,
}

# Validate that probabilities sum to 1
assert np.isclose(
    sum(oven_function_probabilities.values()),
    1.0
), "OVEN_FUNCTIONS probabilities must sum to 1."

oven_function_probabilities

{'V001': 0.04,
 'V002': 0.07,
 'V003': 0.07,
 'V004': 0.1,
 'V005': 0.12,
 'V006': 0.12,
 'V007': 0.1,
 'V008': 0.1,
 'V009': 0.08,
 'V010': 0.08,
 'V011': 0.06,
 'V012': 0.06}